In [1]:
%%writefile tests/test_bank_risk.py
import numpy as np
import pytest

from simulations.bank_risk import risk_metrics, simulate_bank_losses

TWO_REGIMES = {
    "good": {"prob": 0.8, "p_default": 0.01, "p_up": 0.51},
    "bad": {"prob": 0.2, "p_default": 0.06, "p_up": 0.46},
}
INDEPENDENT = {"all": {"prob": 1.0, "p_default": 0.02, "p_up": 0.5}}
PORTFOLIO = dict(n_loans=100, ead=10_000, lgd=0.45, position_value=100_000, n_days=250)


def test_expected_credit_loss_close_to_9000():
    credit = simulate_bank_losses(TWO_REGIMES, **PORTFOLIO, seed=42)["credit"]
    assert credit.mean() == pytest.approx(9_000, rel=0.02)


def test_regimes_fatten_the_tail():
    var_ind = risk_metrics(simulate_bank_losses(INDEPENDENT, **PORTFOLIO, seed=42)["credit"])["VaR"]
    var_reg = risk_metrics(simulate_bank_losses(TWO_REGIMES, **PORTFOLIO, seed=42)["credit"])["VaR"]
    assert var_reg > var_ind


def test_reproducible_with_seed():
    a = simulate_bank_losses(TWO_REGIMES, **PORTFOLIO, n_sims=1_000, seed=7)
    b = simulate_bank_losses(TWO_REGIMES, **PORTFOLIO, n_sims=1_000, seed=7)
    np.testing.assert_array_equal(a["total"], b["total"])


def test_regime_probs_must_sum_to_one():
    bad = {"a": {"prob": 0.5, "p_default": 0.01, "p_up": 0.5}}
    with pytest.raises(ValueError):
        simulate_bank_losses(bad, **PORTFOLIO)


def test_risk_metrics_constant_losses():
    metrics = risk_metrics(np.full(1_000, 500.0))
    assert metrics == pytest.approx({"EL": 500.0, "VaR": 500.0, "ES": 500.0})

Writing tests/test_bank_risk.py


In [3]:
import sys
!{sys.executable} -m pytest tests/test_bank_risk.py -v

============================= test session starts =============================
platform win32 -- Python 3.12.4, pytest-7.4.4, pluggy-1.0.0 -- C:\Users\Adria\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Adria\Desktop\Quant\quant-toolkit\Code\Projects\Project 1 - Bank risk lab
plugins: anyio-4.2.0
collecting ... collected 5 items

tests/test_bank_risk.py::test_expected_credit_loss_close_to_9000 PASSED  [ 20%]
tests/test_bank_risk.py::test_regimes_fatten_the_tail PASSED             [ 40%]
tests/test_bank_risk.py::test_reproducible_with_seed PASSED              [ 60%]
tests/test_bank_risk.py::test_regime_probs_must_sum_to_one PASSED        [ 80%]
tests/test_bank_risk.py::test_risk_metrics_constant_losses PASSED        [100%]

============================== 5 passed in 0.31s ==============================
